# BBC Articles: Collect Full Text for Each Link (Selenium + BeautifulSoup)

This notebook reads `Data.csv`, visits each URL in the `Links` column, extracts the main article text, and writes `Updated_Data.csv` with a new column `Article_Content`.

## Install
```bash
pip install selenium beautifulsoup4 lxml pandas
```

## Notes
- Avoids brittle hashed CSS classes (`sc-...`).
- Uses `<main>`/`<article>` and collects paragraph text.
- Saves at the end.


In [1]:
import time
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


## 1) Load URLs from `Data.csv`

In [2]:
df = pd.read_csv("bbc_business_articles.csv")
df = df.dropna(subset=["url"]).copy()

# Optional: limit for testing
# df = df.head(50).copy()

df["Article_Content"] = ""
print("Rows:", len(df))
df[["url"]].head()


Rows: 24


,url
0,https://www.bbc.com/news/articles/cge89x7re74o
1,https://www.bbc.com/news/articles/cddn6yyr66yo
2,https://www.bbc.com/news/articles/cwy8dxz1g7zo
3,https://www.bbc.com/news/articles/c3ewje4xk3yo
4,https://www.bbc.com/news/articles/c0ljpxek5w2o


## 2) Helper functions

In [3]:
def extract_article_text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")

    main = soup.find("main") or soup
    article = main.find("article") or main

    paras = []
    for p in article.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt:
            continue
        if len(txt) < 30:  # light filtering
            continue
        paras.append(txt)

    text = "\n\n".join(paras)
    return text.replace("�", "'").strip()


In [4]:
def accept_consent_if_present(wait):
    # Best-effort; safe if not present
    for xpath in [
        "//button[contains(.,'Accept') or contains(.,'I agree') or contains(.,'Agree')]",
        "//*[@id='bbccookies-continue-button']",
    ]:
        try:
            wait.until(EC.element_to_be_clickable((By.XPATH, xpath))).click()
            return
        except Exception:
            pass


## 3) Visit each link and extract full text

In [5]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

try:
    for k, (idx, url) in enumerate(df["url"].items(), start=1):
        url = str(url).strip()
        if not url:
            continue

        driver.get(url)
        accept_consent_if_present(wait)

        # Wait for main content
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "main")))
        time.sleep(1.0)  # allow client-side hydration

        html = driver.page_source
        text = extract_article_text_from_html(html)

        df.at[idx, "Article_Content"] = text

        if k % 10 == 0:
            print(f"processed {k}/{len(df)}")

finally:
    driver.quit()


processed 10/24
processed 20/24


## 4) Save to `Updated_Data.csv`

In [6]:
df.to_csv("bbc_Updated_Data.csv", index=False, encoding="utf-8")
print("Saved: Updated_Data.csv")
df[["url", "Article_Content"]].head(3)


Saved: Updated_Data.csv


,url,Article_Content
0,https://www.bbc.com/news/articles/cge89x7re74o,A fire at an oil refinery in Cuba has been bro...
1,https://www.bbc.com/news/articles/cddn6yyr66yo,Andrew Mountbatten-Windsor faces another accus...
2,https://www.bbc.com/news/articles/cwy8dxz1g7zo,Amazon's smart doorbell company is dropping a ...
